## Utils

In [4]:
import asyncio
from typing import Any, Dict, List

import nest_asyncio

from langsmith import Client
from langsmith.evaluation import evaluate
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

from run_pipeline import StaticParams
from src.all_functionality import load_eval_tasks
from src.evaluator import Evaluator

# Allow nested event loops (LangSmith runs its own loop)
nest_asyncio.apply()

# Global setup
EXPERIMENT_PREFIX = "Default. No summary. "
DATASET_NAME = "Default, 6 tasks. No summary."
JUDGE_MODEL_NAME = "gemma27"

client = Client()
core_evaluator = Evaluator(default_judge_model=JUDGE_MODEL_NAME)
eval_tasks = load_eval_tasks("evals")


def _example_to_dict(example: Any) -> Dict[str, Any]:
    if isinstance(example, dict):
        return example
    out: Dict[str, Any] = {}
    out["inputs"] = getattr(example, "inputs", {}) or {}
    out["outputs"] = getattr(example, "outputs", {}) or {}
    return out


def _extract_task_and_state(example_like: Any) -> tuple[str, Dict[str, Any]]:
    ex = _example_to_dict(example_like)
    outputs = ex.get("outputs", {})
    state = outputs.get("pre_computed_state", {})
    task_id = outputs.get("task_id") or ex.get("inputs", {}).get("task_id", "")
    return task_id, state


def _present_score(status_items: List[Dict[str, Any]]) -> tuple[float, int]:
    present_count = sum(
        1
        for item in status_items
        if str(item.get("status", "")).strip().lower() == "present"
    )
    score = (present_count / len(status_items)) if status_items else 0.0
    return score, present_count


class RagStatementsEvaluator:
    """Independent evaluator for statement presence in RAG retrieval context."""

    def __init__(self, evaluator: Evaluator, task_specs: Dict[str, Dict[str, Any]], judge_model_name: str | None = None):
        self.evaluator = evaluator
        self.task_specs = task_specs
        self.judge_model_name = judge_model_name

    async def _compute(self, example: Any) -> Dict[str, Any]:
        task_id, state = _extract_task_and_state(example)
        retrieval_context = state.get("retrieval_context", "")
        statements = self.task_specs.get(task_id, {}).get("correct_statements", [])

        status_items = await self.evaluator.eval_context_statements(
            context=retrieval_context,
            statements=statements,
            judge_model_name=self.judge_model_name,
        )

        score, present_count = _present_score(status_items)

        return {
            "key": "rag_statements_score",
            "score": score,
            "comment": f"task={task_id} present={present_count}/{len(status_items)}",
            "metadata": {"raw": status_items},
        }

    def __call__(self, *, example: Any) -> Dict[str, Any]:
        return asyncio.run(self._compute(example))


class CodeStatementsEvaluator:
    """Independent evaluator for statement presence in generated code."""

    def __init__(self, evaluator: Evaluator, task_specs: Dict[str, Dict[str, Any]], judge_model_name: str | None = None):
        self.evaluator = evaluator
        self.task_specs = task_specs
        self.judge_model_name = judge_model_name

    async def _compute(self, example: Any) -> Dict[str, Any]:
        task_id, state = _extract_task_and_state(example)
        code = state.get("final_code") or state.get("generated_code") or ""
        statements = self.task_specs.get(task_id, {}).get("correct_statements", [])

        status_items = await self.evaluator.eval_context_statements(
            context=code,
            statements=statements,
            judge_model_name=self.judge_model_name,
        )

        score, present_count = _present_score(status_items)

        return {
            "key": "code_statements_score",
            "score": score,
            "comment": f"task={task_id} present={present_count}/{len(status_items)}",
            "metadata": {"raw": status_items},
        }

    def __call__(self, *, example: Any) -> Dict[str, Any]:
        return asyncio.run(self._compute(example))


class CodeExecutionEvaluator:
    """Independent evaluator for generated code execution pass/fail."""

    def __init__(self, evaluator: Evaluator):
        self.evaluator = evaluator

    async def _compute(self, example: Any) -> Dict[str, Any]:
        task_id, state = _extract_task_and_state(example)
        code = state.get("final_code") or state.get("generated_code") or ""

        result = await self.evaluator.eval_code_exec(code)
        execution_pass = bool(result.get("pass", False))

        return {
            "key": "code_execution_score",
            "score": 1.0 if execution_pass else 0.0,
            "comment": f"task={task_id} exec={execution_pass}",
            "metadata": {"raw": result},
        }

    def __call__(self, *, example: Any) -> Dict[str, Any]:
        return asyncio.run(self._compute(example))


# Dummy target: we evaluate pre-computed states only
def dummy_target(inputs: Dict[str, Any]) -> Dict[str, Any]:
    return {"status": "precomputed", "task_id": inputs.get("task_id", "")}


rag_eval = RagStatementsEvaluator(core_evaluator, eval_tasks, judge_model_name=JUDGE_MODEL_NAME)
code_statements_eval = CodeStatementsEvaluator(core_evaluator, eval_tasks, judge_model_name=JUDGE_MODEL_NAME)
code_execution_eval = CodeExecutionEvaluator(core_evaluator)

In [5]:
eval_tasks['add_task']

{'task': 'Add a new task into my Notion Tasks database with the title "Add_task_test", date today, importance of 4 and project ID "31bcb17dcc4480dcb042f86e6a70bb70".',
 'properties': {'title': 'Add_task_test',
  'date': 'today',
  'importance': 4,
  'project_id': '31bcb17dcc4480dcb042f86e6a70bb70'},
 'solution': 'def add_new_task(database_id, title, importance, project_id):\n    import requests\n    import os\n    from datetime import datetime\n    from dotenv import load_dotenv\n    \n    load_dotenv()\n    NOTION_TOKEN = os.getenv("NOTION_TOKEN")\n    HEADERS = {\n        "Authorization": f"Bearer {NOTION_TOKEN}",\n        "Content-Type": "application/json",\n        "Notion-Version": "2022-06-28"\n    }\n    \n    url = "https://api.notion.com/v1/pages"\n    \n    # Parse "today" dynamically into an ISO 8601 string (YYYY-MM-DD)\n    today_date = datetime.now().strftime("%Y-%m-%d")\n    \n    # Schema for creating a new database page with specific property types\n    payload = {\n   

## Samples

In [2]:
# Flexible sample loading + dataset creation in a single cell
# Change these variables directly for each run
THREAD_PREFIX_FILTERS = []  # [] means no filtering
N_LAST_RUNS = 6

static_params = StaticParams()

states_to_evaluate: List[Dict[str, Any]] = []

# Create or fetch dataset
existing = None
for ds in client.list_datasets(dataset_name=DATASET_NAME):
    if ds.name == DATASET_NAME:
        existing = ds
        break
    
if existing:
    print(f"Found existing dataset with name {DATASET_NAME}. To avoid confusion, the code is set to raise an error if the dataset already exists. Please change the DATASET_NAME variable or remove the existing dataset to proceed.")
    raise ValueError("Dataset already exists. Please change DATASET_NAME or remove the existing dataset to proceed.")

dataset = client.create_dataset(dataset_name=DATASET_NAME)


async with AsyncSqliteSaver.from_conn_string(static_params.sqlite_saver_path) as checkpointer:
    # Gather unique thread IDs from checkpoint store
    thread_ids: List[str] = []
    seen_thread_ids = set()

    async for checkpoint in checkpointer.alist(None):
        thread_id = checkpoint.config.get("configurable", {}).get("thread_id")

        if not thread_id or thread_id in seen_thread_ids:
            continue

        if THREAD_PREFIX_FILTERS and not any(thread_id.startswith(prefix + "_") for prefix in THREAD_PREFIX_FILTERS):
            continue

        seen_thread_ids.add(thread_id)
        thread_ids.append(thread_id)

    # Most recent are generally appended later; keep this direct and easy to tweak
    selected_thread_ids = thread_ids[:N_LAST_RUNS]

    for thread_id in selected_thread_ids:
        config = {"configurable": {"thread_id": thread_id}}
        checkpoint_tuple = await checkpointer.aget_tuple(config)
        if not checkpoint_tuple:
            continue

        channel_values = checkpoint_tuple.checkpoint.get("channel_values", {})
        task_id = thread_id.rsplit("_", 1)[0] if "_" in thread_id else thread_id

        states_to_evaluate.append({
            "thread_id": thread_id,
            "task_id": task_id,
            "query": channel_values.get("user_prompt", ""),
            "pre_computed_state": channel_values,
        })

# Push examples
for state in states_to_evaluate:
    client.create_example(
        inputs={
            "query": state["query"],
            "task_id": state["task_id"],
            "thread_id": state["thread_id"],
        },
        outputs={
            "task_id": state["task_id"],
            "pre_computed_state": state["pre_computed_state"],
        },
        dataset_id=dataset.id,
    )

print(f"Dataset: {dataset.name} | uploaded examples: {len(states_to_evaluate)}")
states_to_evaluate[:2]

Dataset: Default, 6 tasks. No summary. | uploaded examples: 6


[{'thread_id': 'update_task_status_synthetic_20260308141242',
  'task_id': 'update_task_status_synthetic',
  'query': 'Update the task with id 31acb17dcc4480edbe0bcc0079323783 by changing its "Status" to "Done" and clearing the date.',
  'pre_computed_state': {'user_prompt': 'Update the task with id 31acb17dcc4480edbe0bcc0079323783 by changing its "Status" to "Done" and clearing the date.',
   'retrieval_context': '| `property`                                                                                                                                                                                                                         | `string` | The name of the property as it appears in the data source, or the property ID.                                                                                                                                                                | `"Task completed"`               |\n| `checkbox`<br />`date`<br />`files`<br />`formula`<br />`mult

In [32]:
for st in states_to_evaluate:
    print(f"Thread: {st['thread_id']} | Task: {st['task_id']} | Query: {st['query'][:50]}...")

Thread: update_task_status_synthetic_20260307131627 | Task: update_task_status_synthetic | Query: Update the task with id 31acb17dcc4480edbe0bcc0079...
Thread: retrieve_tasks_20260307131325 | Task: retrieve_tasks | Query: Retrieve the last 3 tasks from my Notion Tasks dat...
Thread: get_page_content_synthetic_20260307131057 | Task: get_page_content_synthetic | Query: Retrieve all the content and blocks from the inbox...
Thread: append_checklist_to_page_synthetic_20260307130844 | Task: append_checklist_to_page_synthetic | Query: Add a checklist to the inbox page containing the i...
Thread: add_toggle_to_page_20260307130801 | Task: add_toggle_to_page | Query: Add a new toggle to my Notion page with the title ...
Thread: add_task_20260307130715 | Task: add_task | Query: Add a new task into my Notion Tasks database with ...


## Evaluate

In [5]:
results = evaluate(
    dummy_target,
    data=DATASET_NAME,
    evaluators=[rag_eval, code_statements_eval, code_execution_eval],
    experiment_prefix=EXPERIMENT_PREFIX,
)
results

View the evaluation results for experiment: 'Default. No summary. -31f4680d' at:
https://smith.langchain.com/o/67129af7-abb6-42fb-b044-bbe6030db476/datasets/3c89792e-9b83-4d8a-b89a-b3e78f514523/compare?selectedSessions=672eead2-bd47-4d42-82ca-7ba26cfa7a05




0it [00:00, ?it/s]

2026-03-08 15:04:05 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-27b-it, finish_reason: stop


2026-03-08 15:04:16 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


...

2026-03-08 15:06:12 | INFO     | src.evaluator | eval_code_exec | pass=False
2026-03-08 15:06:21 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-27b-it, finish_reason: stop


2026-03-08 15:06:33 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-08 15:06:33 | INFO     | src.evaluator | eval_code_exec | pass=True


[async_chat_wrapper] Model: gemma-3-27b-it, finish_reason: stop


,inputs.query,inputs.task_id,inputs.thread_id,outputs.status,outputs.task_id,error,reference.task_id,reference.pre_computed_state,feedback.rag_statements_score,feedback.code_statements_score,feedback.code_execution_score,execution_time,example_id,id
0,Add a new task into my Notion Tasks database w...,add_task,add_task_20260308140952,precomputed,add_task,None,add_task,"{'passed': False, 'trials': [], 'verdict': {},...",0.333333,0.000000,0.0,0.006484,9eecc08e-211a-474a-b17e-f1c0e8aea0b2,019ccd55-2228-7b52-b5af-a413480b5146
1,Add a new toggle to my Notion page with the ti...,add_toggle_to_page,add_toggle_to_page_20260308141018,precomputed,add_toggle_to_page,None,add_toggle_to_page,"{'passed': False, 'trials': [], 'verdict': {},...",0.666667,1.000000,0.0,0.000437,0d25f153-430e-4e87-a7e9-e7c0dfdaf8b4,019ccd55-7d1e-7b12-a67e-1f2be02ed49d
2,Add a checklist to the inbox page containing t...,append_checklist_to_page_synthetic,append_checklist_to_page_synthetic_20260308141112,precomputed,append_checklist_to_page_synthetic,None,append_checklist_to_page_synthetic,"{'passed': False, 'trials': [], 'verdict': {},...",0.166667,0.000000,0.0,0.000225,5cba9a8a-0bd0-46b0-bdfd-5ba4a862c494,019ccd55-ceb6-7901-83c1-07b87f3c245f
3,Retrieve all the content and blocks from the i...,get_page_content_synthetic,get_page_content_synthetic_20260308141134,precomputed,get_page_content_synthetic,None,get_page_content_synthetic,"{'passed': False, 'trials': [], 'verdict': {},...",0.166667,0.666667,0.0,0.000412,ef090357-51b5-487a-917c-1c05d530e410,019ccd56-75b0-7581-ab22-72fa36ec23c0
4,Retrieve the last 3 tasks from my Notion Tasks...,retrieve_tasks,retrieve_tasks_20260308141222,precomputed,retrieve_tasks,None,retrieve_tasks,"{'passed': False, 'trials': [], 'verdict': {},...",0.666667,0.666667,0.0,0.000220,9865d56e-9b93-48c7-8fec-fe6ae1f2778f,019ccd56-e00e-7383-bcf8-d0fcb405b23b
5,Update the task with id 31acb17dcc4480edbe0bcc...,update_task_status_synthetic,update_task_status_synthetic_20260308141242,precomputed,update_task_status_synthetic,None,update_task_status_synthetic,"{'passed': False, 'trials': [], 'verdict': {},...",0.000000,0.500000,1.0,0.000383,7e94e8d9-e2f6-4d94-8bb9-82dc1077b692,019ccd57-3ed3-7640-aeb0-7e4c0c4b8353


In [53]:
for st in states_to_evaluate[1:2]:
    res = await core_evaluator.eval_context_statements(
        context=st["pre_computed_state"].get("generated_code", ""),
        statements=eval_tasks.get(st["task_id"], {}).get("correct_statements", []),
    )
    print(res)

2026-03-08 13:55:09 | INFO     | httpx | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"


[async_chat_wrapper] Model: gemma-3-27b-it, finish_reason: stop
[{'statement': 'Endpoint is POST https://api.notion.com/v1/databases/{database_id}/query for querying a database.', 'status': 'Wrong', 'evidence': 'url = f"https://api.notion.com/v1/databases/query"', 'reasoning': 'The endpoint is missing the database_id path parameter.'}, {'statement': "Header 'Notion-Version: 2022-06-28' is required on every request.", 'status': 'Present', 'evidence': 'headers = {\n        "Authorization": f"Bearer {os.getenv(\'NOTION_TOKEN\')}",\n        "Notion-Version": "2022-06-28"\n    }', 'reasoning': "The header 'Notion-Version: 2022-06-28' is present in the headers dictionary."}, {'statement': "Result count is limited via the 'page_size' key in the request body (e.g. 'page_size': 3).", 'status': 'Present', 'evidence': '"page_size": num_tasks', 'reasoning': "The 'page_size' key is used in the request body to limit results."}, {'statement': "Sorting is defined under a 'sorts' array: [{'property': '